## Script to extract Hansen et al. (2013) forest loss for different buffer sizes from point locations

In [ ]:
## Variables to set
project_id = '' # google cloud project containing asset locations at asset_path
working_dir = '' # name for directory containing assets
working_dir_path = f'projects/{project_id}/assets/{working_dir}' # cloud directory containing asset and in which to create output directory
asset_path = f'{working_dir_path}/xyyear' # asset containing x, y, and loan signed year for each project-loan pair

In [ ]:
import ee
import os
import re
from dotenv import load_dotenv
import datetime
from google.cloud import storage
from google.cloud.storage import Client, transfer_manager
import json
import geopandas as gpd
import pandas as pd
import numpy as np
import geemap
import zipfile
import fiona
# Load environment variables from .env file
load_dotenv("credentials/.env")

## AUTHENTICATION ##
# authenticate with google earth engine for analysis

def authenticate_gee():
    # Service account email
    service_account = os.environ.get('SERVICE_ACCOUNT_EMAIL')

    # Path to the service account key file inside the container
    credentials_path = 'credentials/ee_service_account.json'

    # Initialize Earth Engine with the service account credentials
    credentials = ee.ServiceAccountCredentials(service_account, credentials_path)
    ee.Initialize(credentials)
    
def create_buffers(feature:ee.Feature, buffer_distances:list):
    def mapDistance(distance):
        uid = ee.String(feature.getNumber('Longitude')).cat(ee.String('x')).cat(feature.getNumber('Latitude')).cat(ee.String(distance))
        uid.name = lambda: 'uniqueID'
        return feature.buffer(distance).set({
            'BUFFER_RADIUS': distance,
            'id': uid
            })
    return ee.FeatureCollection(
        buffer_distances.map(mapDistance)
        )

# Function to export a FeatureCollection to Earth Engine asset, Google Drive, or BigQuery
def export_feature_collection(feature_collection, description, export_type, asset_id=None, drive_folder=None, bigquery_dataset=None, table_name=None, file_format='CSV', selectors=None):
    # Create the initial configuration dictionary for the export task
    task_config = {
        'collection': feature_collection,  # The FeatureCollection to export
        'description': description,        # Description for the export task
    }

    # If selectors are provided, add them to the configuration
    if selectors is not None:
        task_config['selectors'] = selectors

    # Determine the export type and configure accordingly
    if export_type == 'toAsset':
        # Export to Earth Engine asset
        if asset_id is None:
            raise ValueError("asset_id must be provided for export to an Earth Engine asset.")
        task_config['assetId'] = asset_id  # Set the asset ID where the data will be saved
        task = ee.batch.Export.table.toAsset(**task_config)  # Create the export task for asset
    elif export_type == 'toDrive':
        # Export to Google Drive
        if drive_folder is not None:
            task_config['folder'] = drive_folder  # Set the Google Drive folder for export
        task_config['fileFormat'] = file_format  # Specify the file format (default is CSV)
        task = ee.batch.Export.table.toDrive(**task_config)  # Create the export task for Drive
    elif export_type == 'toBigQuery':
        # Export to BigQuery
        if bigquery_dataset is None or table_name is None:
            raise ValueError("bigquery_dataset and table_name must be provided for export to BigQuery.")
        task_config['dataset'] = bigquery_dataset  # Set the BigQuery dataset
        task_config['table'] = table_name  # Set the BigQuery table name
        task_config['append'] = True
        task_config['overwrite'] = False

        task = ee.batch.Export.table.toBigQuery(**task_config)  # Create the export task for BigQuery
    else:
        # Raise an error if the export type is not recognized
        raise ValueError("export_type must be one of 'toAsset', 'toDrive', or 'toBigQuery'.")
    
    task.start()  # Start the export task

def map_disasters(buffered_asset_features, disasters, year):
    """
    Maps disasters to the asset based on the year of the disaster and 
    returns a feature collection with one feature per disaster or no disaster,
    maintaining the original feature's geometry and adding a boolean column.

    Args:
        buffered_asset_feature: FeatureCollection of the original features.
        disasters: FeatureCollection of disasters.
        year: The year to filter disasters on.
        disaster_properties: List of disaster properties to retain, with a 'disaster_' prefix.
    """
    # Pre-filter disasters to reduce data volume early
    filtered_disasters = disasters.filter(
        ee.Filter.eq('year', ee.Number(year).format())
    )
    
    # Define spatial filter once
    spatial_filter = ee.Filter.intersects(
        leftField='.geo',
        rightField='.geo',
        maxError=10
    )
       # Define property mapping for disaster attributes
    disaster_property_mapping = {
        'adm1': 'disaster_adm1',
        'adm2': 'disaster_adm2',
        'adm3': 'disaster_adm3',
        'country': 'disaster_country',
        'disasterno': 'disaster_disasterno',
        'disasterty': 'disaster_disasterty',
        'geo_id': 'disaster_geo_id',
        'geolocatio': 'disaster_geolocatio',
        'gwno': 'disaster_gwno',
        'hist_count': 'disaster_hist_count',
        'historical': 'disaster_historical',
        'id': 'disaster_id',
        'iso3': 'disaster_iso3',
        'latitude': 'disaster_latitude',
        'level': 'disaster_level',
        'location': 'disaster_location',
        'longitude': 'disaster_longitude',
        'year': 'disaster_year'
    }
    
    def map_disaster_properties(disaster):
        """Maps original disaster properties to prefixed properties efficiently"""
        feature = ee.Feature(disaster)
        return feature.select(
            list(disaster_property_mapping.keys()),
            list(disaster_property_mapping.values()),
            False
        )
    def process_feature(feature):
        """Processes each feature with its associated disasters"""
        feature = ee.Feature(feature)
        disasters_list = ee.List(feature.get('disasters'))
        
        def create_disaster_feature(disaster):
            """Creates a feature with disaster properties"""
            mapped_disaster = map_disaster_properties(disaster)
            return ee.Feature(feature.geometry()) \
                .copyProperties(feature, exclude=['disasters']) \
                .copyProperties(mapped_disaster) \
                .set('disaster_occurred', True)
        
        # Create features with disasters
        with_disasters = ee.FeatureCollection(
            disasters_list.map(create_disaster_feature)
        ).filter(ee.Filter.neq('disaster_occurred', None))
        
        # Create no-disaster feature
        without_disaster = ee.FeatureCollection([
            ee.Feature(feature.geometry())
            .copyProperties(feature, exclude=['disasters'])
            .set('disaster_occurred', False)
            .set('list_size', disasters_list.size())
        ]).filter(ee.Filter.eq('list_size', 0))
        
        # Merge the filtered collections
        return with_disasters.merge(without_disaster)
    
    # Apply join and process features
    joined_features = ee.Join.saveAll(
        matchesKey='disasters',
        outer=True
    ).apply(
        buffered_asset_features,
        filtered_disasters,
        spatial_filter
    )

    #DEBUG
    # feature = ee.Feature(joined_features.toList(20).get(2))
    # process_feature(feature)

    # Process and return final collection
    return ee.FeatureCollection(joined_features) \
        .map(process_feature) \
        .flatten()

def calculate_annual_metrics(year, buffered_assets, tileScale=16, resolution=30):
    """
    For a single year, calculates area (ha) of forest loss, forest gained, burned forest and forest remaining
    within each buffered feature. Returns a FeatureCollection with stats.
    Also calculates starting forest cover for each year.
    """

    year_offset = ee.Number(year).subtract(2000)
  
    # If year > 2000, subtract all forest loss from previous years
    previous_losses_mask = lossyear.gt(0).And(lossyear.lt(year_offset))
    forest_cover_current_yr_thresholded = forest_cover_2000_thresholded.where(previous_losses_mask.eq(1),0).selfMask()

    
    # year_metrics is a three-band image now:
    #   band0: forest cover at start of year (ha)  
    #   band1: 1 where this pixel lost forest in 'year'; otherwise masked out
    #   band2: 1 where pixel burned in 'year'; otherwise masked out
    # Then each band is multiplied by pixel area and scaled to hectares.
    year_metrics = ee.Image.cat([
        forest_cover_current_yr_thresholded,
        forest_cover_current_yr_thresholded.updateMask(lossyear.eq(year_offset)).rename('FOREST_LOSS_HA'),
        forest_cover_current_yr_thresholded.updateMask(lossyear.eq(year_offset)) 
            .updateMask(burned_area.filter(ee.Filter.calendarRange(year, year, 'year')).max().gt(0)).rename('BURN_LOSS_HA')
    ]).multiply(ee.Image.pixelArea()).divide(10000) # Convert to hectares

    # Get initial stats
    area_stats = year_metrics.reduceRegions(
        collection = buffered_assets,
        reducer = ee.Reducer.sum(),  # Ensure the reducer is specified
        scale = resolution,
        tileScale = tileScale,
        crs = 'EPSG:4326',
    )
    
    # Attach the year as a property 
    area_stats = area_stats.map(lambda f: f.set('ANALYSIS_YEAR', year))

    area_stats = map_disasters(area_stats, gdis, year)

    return ee.FeatureCollection(area_stats)

def extract(region, all_assets, year_range, tileScale, resolution, buffer_distances):
    """
    Filters the assets to those within 'region', buffers them, then
    calculates annual metrics across 'year_range'. Returns a FeatureCollection.
    """
    geo = region.geometry()
    subset = all_assets.filterBounds(geo)
    return ee.FeatureCollection(year_range.map(lambda y: calculate_annual_metrics(
            y,
            subset.map(lambda feat: create_buffers(feat, buffer_distances)).flatten(),
            tileScale,
            resolution
        ))).flatten()

In [ ]:
if __name__ == '__main__':
    authenticate_gee()  # Login / service account init

    # Settings
    start_year, end_year = 2001, 2023
    years = list(range(start_year, end_year + 1))
    resolution = 500  # can we take this down to native resolution? How does that impact burn data? Also burn data, is vegetation burn. Also do we need the forest threshold, or do they do it on their side. Send the data to Theo
    tileScale = 16
    year_batch_size = 1
    grid_scale = 1  # in degrees
    grid_density = 1000 # a measure of how many assests to assess in each grid cell, as part of each reducer process
    buffer_distances = ee.List([1000, 5000, 10000, 20000, 30000, 40000, 50000, 60000, 70000, 80000, 90000, 100000])
    all_assets = ee.FeatureCollection(asset_path)
    
    # Global Hansen data
    gfc = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
    treecover2000 = gfc.select('treecover2000')
    lossyear = gfc.select('lossyear') 
    
    # referenced by calculate_annual_metrics
    forest_threshold = 25
    forest_cover_2000_thresholded = treecover2000.gt(forest_threshold).selfMask().rename('FOREST_COVER_HA') # Pixels with treecover > threshold in year 2000
    
    # Burned area data
    burned_area = ee.ImageCollection('MODIS/061/MCD64A1')  # also referenced globally
    burned_area = burned_area.select('BurnDate')
    
    # Build bounding box around the assets, then create a grid at n-degree spacing
    bbox = all_assets.geometry().bounds()
    proj = ee.Projection('EPSG:4326').scale(grid_scale, grid_scale)
    regions = bbox.coveringGrid(proj)
    
    # Load disaster dataset from gee community catalogue
    gdis = ee.FeatureCollection("projects/sat-io/open-datasets/gdis_1960-2018")

    def export_year_range(year_range, description_suffix, asset_id_suffix):
        years_fc = regions.map(
            lambda r: extract(r, all_assets, year_range, tileScale, resolution, buffer_distances)
        ).flatten()
        export_feature_collection(
            feature_collection=years_fc,
            description=f'loans_gridScale_{str(grid_scale).replace(".", "_")}_tileScale_{tileScale}_scale_{resolution}_{description_suffix}_1k_100k_buffer',
            export_type='toAsset',
            asset_id=f'{working_dir_path}/output/loans_gridScale_{str(grid_scale).replace(".", "_")}_tileScale_{tileScale}_scale_{resolution}_{asset_id_suffix}_1k_100k_buffer',
        )
        print(f"Export task for {description_suffix} started.")

    Map = geemap.Map()
    Map.addLayer(treecover2000, {}, 'tree cover 2000')
    Map.addLayer(gdis, {}, 'disasters')
    Map.addLayer(all_assets, {}, 'assets')
    display(Map)
    

In [ ]:
## Export deforestaion across buffers for each year (where year_batch_size=1)
year_batches = [years[i:i+year_batch_size] for i in range(0, len(years), year_batch_size)]
for batch in year_batches:
    description_suffix = f"yearRange_{batch[0]}_to_{batch[-1]}"
    asset_id_suffix = f"yearRange_{batch[0]}_to_{batch[-1]}"
    export_year_range(ee.List(batch), description_suffix, asset_id_suffix)

In [ ]:
authenticate_gee()
def get_asset_list(parent):
    parent_asset = ee.data.getAsset(parent)
    parent_id = parent_asset['name']
    parent_type = parent_asset['type']
    asset_list = []
    child_assets = ee.data.listAssets({'parent': parent_id})['assets']
    for child_asset in child_assets:
        child_id = child_asset['name']
        child_type = child_asset['type']
        if child_type in ['FOLDER','IMAGE_COLLECTION']:
            # Recursively call the function to get child assets
            asset_list.extend(get_asset_list(child_id))
        else:
            asset_list.append(child_id)
    return asset_list
#
out_dir = f'{working_dir_path}/output'
out_list = get_asset_list(out_dir)

In [ ]:
df_all = []
for out_file in out_list:
    print('processing', out_file)
    df = geemap.ee_to_df(ee.FeatureCollection.loadTable(out_file)).set_index('id')
    for i in range(df.shape[0]):
        idx = df.iloc[i].name
        buffer = df.iloc[i].BUFFER_RADIUS
        df.at[idx, f'forest_cover_{buffer//1e3:.0f}km'] = df.iloc[i].FOREST_COVER_HA
        df.at[idx, f'forest_loss_{buffer//1e3:.0f}km'] = df.iloc[i].FOREST_LOSS_HA
        df.at[idx, f'burn_loss_{buffer//1e3:.0f}km'] = df.iloc[i].BURN_LOSS_HA
    df = df.groupby(['ANALYSIS_YEAR', 'ProjectID']).first().reset_index()
    df_all += [df.copy()]

df_all = pd.concat(df_all)

columns = ['year_signi']
for buffer in np.sort(np.unique(df_all.BUFFER_RADIUS)):
    columns += [f'forest_cover_{buffer*1e-3:.0f}km', f'forest_loss_{buffer*1e-3:.0f}km', f'burn_loss_{buffer*1e-3:.0f}km']

df_all.set_index(['ProjectID', 'ANALYSIS_YEAR']).sort_index()[columns].to_csv('output/loans_forestLoss_merged.csv')